In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os
os.chdir('/Users/lior/DS6021/DS6021_F26/data')
 
penguins = pd.read_csv('penguins.csv')
peng_subset = penguins.dropna(subset=['bill_depth_mm', 'body_mass_g'])
 
x = peng_subset['bill_depth_mm'].to_numpy()
y = peng_subset['body_mass_g'].to_numpy()
n = len(x)

In [3]:
#2a & 2b
X = sm.add_constant(peng_subset['bill_depth_mm'])
results = sm.OLS(peng_subset['body_mass_g'], X).fit()
print(results.summary())
 
analytic_ci = results.conf_int()
print("\n95% CIs from statsmodels:")
print(analytic_ci)

                            OLS Regression Results                            
Dep. Variable:            body_mass_g   R-squared:                       0.223
Model:                            OLS   Adj. R-squared:                  0.220
Method:                 Least Squares   F-statistic:                     97.41
Date:                Sat, 12 Sep 2026   Prob (F-statistic):           2.28e-20
Time:                        13:12:45   Log-Likelihood:                -2728.7
No. Observations:                 342   AIC:                             5461.
Df Residuals:                     340   BIC:                             5469.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const          7488.6524    335.218     22.340

2a
The standard error of the slope was 19.417, the t-statistic was -9.870 and the p-value was less than 0.001. With those values, we can reject the hypothesis that bill depth has no linear association with body mass since theres a very clear sign of a negative linear regression between the bill depth and the body mass.

2b
The r^2 value is 0.223 and the p value for β₁ is approximately 0. The two values are consistent with the number of observations because when there are larger values it makes it a lot easier to see when the slope is different from zero. But even though bill depth is statistically there, its a weak factor to go off of.

In [8]:
#2c

rng = np.random.default_rng(42)
n_boot =10000
boot = np.empty((n_boot, 2))
 
for i in range(n_boot):
    idx = rng.integers(0, n, n)# resample
    xb, yb = x[idx],y[idx]
    xm, ym = xb.mean(),yb.mean()
    slope = ((xb-xm) * (yb- ym)).sum() /((xb- xm) ** 2).sum()
    boot[i] = (ym - slope * xm, slope)
 
boot_df = pd.DataFrame(boot, columns=['b0', 'b1'])
 
comparison = pd.DataFrame({
    'CI low (statsmodels)':analytic_ci[0].to_numpy(),
    'CI low (bootstrap)':[boot_df['b0'].quantile(.025), boot_df['b1'].quantile(.025)],
    'CI high (statsmodels)':analytic_ci[1].to_numpy(),
    'CI high (bootstrap)':[boot_df['b0'].quantile(.975), boot_df['b1'].quantile(.975)],
}, index=['b0', 'b1'])
 
comparison['width (statsmodels)'] = comparison['CI high (statsmodels)'] - comparison['CI low (statsmodels)']
comparison['width (bootstrap)'] = comparison['CI high (bootstrap)'] - comparison['CI low (bootstrap)']
 
print("\nComparison of statsmodels vs. bootstrap 95% CIs:")
print(comparison.round(2).T)


Comparison of statsmodels vs. bootstrap 95% CIs:
                            b0      b1
CI low (statsmodels)   6829.29 -229.84
CI low (bootstrap)     6959.09 -223.88
CI high (statsmodels)  8148.01 -153.45
CI high (bootstrap)    8059.26 -161.91
width (statsmodels)    1318.72   76.39
width (bootstrap)      1100.17   61.97


2c - a
The bootstrap gives the narrower interval which tells us that there are plot deviations and skews which tends to increase the standard errors estimate compared to what the resampling actually observes

2c - b
It assumes that its normally distributed. The bootstrap doesnt make any distributional assumotions, it only estimates the sampling variability.


In [9]:
#2c - c

def run_bootstrap(B):
    boot_rng = np.random.default_rng() 
    out = np.empty((B, 2))
    for i in range(B):
        idx = boot_rng.integers(0, n, n)
        xb, yb = x[idx],y[idx]
        xm, ym = xb.mean(), yb.mean()
        slope = ((xb- xm)*(yb- ym)).sum() /((xb - xm) **2).sum()
        out[i] = (ym - slope * xm, slope)
    return pd.DataFrame(out, columns=['b0', 'b1'])


 
for B in [100, 1000, 10000]:
    rows =[]
    for rep in range(10):
        bdf = run_bootstrap(B)
        rows.append({
            'rep': rep + 1,
            'b0_2.5%':bdf['b0'].quantile(0.025),
            'b0_97.5%':bdf['b0'].quantile(0.975),
            'b1_2.5%':bdf['b1'].quantile(0.025),
            'b1_97.5%':bdf['b1'].quantile(0.975),
        })
    table = pd.DataFrame(rows)
    print(f"\n=== B = {B}: 10 repeated bootstrap simulations ===")
    print(table.round(2).to_string(index=False))


=== B = 100: 10 repeated bootstrap simulations ===
 rep  b0_2.5%  b0_97.5%  b1_2.5%  b1_97.5%
   1  6958.56   8057.31  -224.90   -162.88
   2  6924.44   8060.19  -220.40   -159.49
   3  7001.62   8213.81  -232.53   -164.94
   4  6923.85   8037.53  -221.42   -159.80
   5  6953.51   7990.32  -222.42   -162.99
   6  7008.32   8011.46  -220.92   -163.66
   7  7014.34   7967.37  -219.22   -165.09
   8  6941.74   8038.66  -221.75   -161.25
   9  6969.96   7985.28  -221.24   -161.99
  10  7083.13   8228.29  -234.90   -168.08

=== B = 1000: 10 repeated bootstrap simulations ===
 rep  b0_2.5%  b0_97.5%  b1_2.5%  b1_97.5%
   1  6914.78   8016.71  -221.38   -159.15
   2  6972.67   8076.19  -224.67   -162.87
   3  6941.87   8021.87  -221.92   -162.29
   4  6959.24   8018.32  -222.57   -162.28
   5  6963.11   8019.55  -222.08   -162.54
   6  6992.16   8055.01  -223.62   -163.80
   7  6984.41   8037.80  -223.69   -163.87
   8  6959.37   8009.99  -220.71   -162.37
   9  6958.48   8009.01  -221.59   

2c - d

it tells us that a larger value B doesnt give a better estimate of the uncertainty that actually depends on your sample size, but instead allows the estimate of the CI to be more reproducable and therefore reliable.
